## Import necessary libraries

In [4]:
import re
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed

In [5]:
# Base URL
BASE_URL = 'https://www.hotelchains.com'

# URLs to scrape
urls = [
    'https://www.hotelchains.com/canada/',
    'https://www.hotelchains.com/usa/'
]

def scrape_hotels(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    rows = soup.select('table tbody tr')

    results = []
    for row in rows:
        # Skip hidden rows
        # if 'style' in row.attrs and 'display:none' in row.attrs['style']:
        #     continue
        a_tag = row.find('a')
        if a_tag:
            state = a_tag.text.strip()
            relative_link = a_tag['href'].strip()
            full_link = BASE_URL + relative_link
            results.append({'state': state, 'link': full_link})
    return results

# Collect data from both URLs
all_data = []
for url in urls:
    print(f"Scraping: {url}")
    all_data.extend(scrape_hotels(url))

# Save to CSV
csv_filename = 'hotels_data_links.csv'
with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['state', 'link'])
    writer.writeheader()
    writer.writerows(all_data)

print(f"\n✅ Data saved to '{csv_filename}'")


Scraping: https://www.hotelchains.com/canada/
Scraping: https://www.hotelchains.com/usa/

✅ Data saved to 'hotels_data_links.csv'


In [6]:
BASE_URL = "https://www.hotelchains.com"

def extract_country_from_url(url):
    # Assuming URL like https://www.hotelchains.com/canada/cityname/
    parts = url.split('/')
    for part in parts:
        if part.lower() in ['canada', 'usa', 'mexico', 'uk', 'france']:  # add more if needed
            return part.capitalize()
    return ""

def scrape_hotels_from_url(state, url):
    country = extract_country_from_url(url)
    hotels = []
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        rows = soup.select("table.hotel-list tbody tr[data-name]")
        for row in rows:
            h5 = row.select_one("td h5")
            if not h5:
                continue
            name = h5.contents[0].strip() if h5.contents else ""

            small = h5.select_one("small")
            address_full = small.get_text(strip=True) if small else ""

            parts = address_full.split(",")
            address = parts[0].strip() if parts else ""
            state_from_address = parts[1].strip() if len(parts) > 1 else ""

            a_tag = row.select_one("a[href*='/r/']")
            booking_path = a_tag['href'] if a_tag else ""
            booking_url = urljoin(BASE_URL, booking_path) if booking_path else ""

            hotels.append({
                "name": name,
                "address": address,
                "state": state_from_address if state_from_address else state,
                "country": country,
                "booking_url": booking_url
            })

    except Exception as e:
        print(f"Error scraping {url}: {e}")

    return hotels

def main():
    input_csv = "hotels_data_links.csv"
    output_csv = "hotels.csv"
    all_hotels = []

    # Read input CSV
    with open(input_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    # Use ThreadPoolExecutor for concurrent scraping
    with ThreadPoolExecutor(max_workers=8) as executor:
        future_to_row = {executor.submit(scrape_hotels_from_url, row['state'], row['link']): row for row in rows}

        for future in as_completed(future_to_row):
            result = future.result()
            if result:
                all_hotels.extend(result)

    # Write all data to output CSV
    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        fieldnames = ["name", "address", "state", "country", "booking_url"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_hotels)

    print(f"Scraped data saved to {output_csv}")

if __name__ == "__main__":
    main()


Scraped data saved to hotels.csv


In [7]:
input_csv = "hotels.csv"
output_csv = "hotels_cleaned.csv"

def clean_whitespace(text):
    return re.sub(r'\s+', ' ', text.strip())

# Read, clean, and write
with open(input_csv, newline='', encoding='utf-8') as infile, \
     open(output_csv, mode='w', newline='', encoding='utf-8') as outfile:

    reader = csv.DictReader(infile)
    fieldnames = reader.fieldnames
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    writer.writeheader()
    for row in reader:
        row['address'] = clean_whitespace(row.get('address', ''))
        writer.writerow(row)

print(f"Cleaned addresses saved to {output_csv}")


Cleaned addresses saved to hotels_cleaned.csv
